## 1. Configuration and Imports

In [ ]:
import json
import time
import numpy as np
from numpy.linalg import norm
from pydantic import BaseModel, Field
from ollama import chat, embeddings
from pathlib import Path
from rank_bm25 import BM25Okapi
from concurrent.futures import ThreadPoolExecutor

# ----- VARIABLES -----
model = "llama3:latest" #
ratio_queries = 1 # (0.1 = 10% of the dataset)
embedding_model = "mxbai-embed-large"
queries_type = "tool_only" #tool_param_all tool_only


# ----- FILE PATHS -----
script_folder = Path().absolute()
tools_list_path = script_folder.parent / 'input' / 'tools' / 'tools_list_exemples.json'
queries_list_path = script_folder.parent / 'input' / 'user_queries' / f'{queries_type}.json'
results_path = script_folder / 'output' / f'{model.replace(":", "-")}_{queries_type}_bm25.json'

## 2. RAG Engine (Retrieval-Augmented Generation)

In [15]:
def get_vector(text): 
    """Fetches the embedding vector for a given text from Ollama."""
    response = embeddings(model=embedding_model, prompt=text)
    return response['embedding']

def precompute_tool_embeddings(tools_list):
    """Pre-calculates embeddings for all tools to save time during the routing phase."""
    print("Caching tool embeddings...")
    vecs = []
    for tool in tools_list:
        name = tool.get('name','')
        desc = tool.get('description',str(tool))
        tags = ", ".join(tool.get('tags',[]))
        examples = " ".join(tool.get('examples',[]))
        text_to_embed = f"{name} : {desc}. Tags {tags}. Exemples {examples}."
        vecs.append(get_vector(text_to_embed))
    return np.array(vecs)

def precompute_bm25_cache(tool_list):
    """Pre-calculates cache for all tools to save time during the routing phase."""
    tokenized_corpus = []
    for tool in tool_list:
        text_for_bm25 = f"{tool.get('name', '')} {' '.join(tool.get('tags', []))}".lower()
        tokenized_corpus.append(text_for_bm25.split(" "))
    return BM25Okapi(tokenized_corpus)

def retrieve_top_tools(user_prompt, tools_list, tool_matrix, bm25_index, top_k):
    """Performs a fast vectorized cosine similarity search to find the top K tools."""

    # 1. Vector Score (Numpy)
    query_vec = np.array(get_vector(user_prompt))
    vector_scores = np.dot(tool_matrix, query_vec) / (norm(tool_matrix, axis=1) * norm(query_vec))
    
    # 2. Lexical Score (BM25)
    tokenized_query = user_prompt.lower().split(" ")
    lexical_scores = bm25_index.get_scores(tokenized_query)

    # 3. Normalization
    vector_scores_norm = (vector_scores - np.min(vector_scores)) / (np.max(vector_scores) - np.min(vector_scores) + 1e-9)
    lexical_scores_norm = (lexical_scores - np.min(lexical_scores)) / (np.max(lexical_scores) - np.min(lexical_scores) + 1e-9)
    
    # 4. combination (70% 30%)
    final_scores = (vector_scores_norm * 1) + (lexical_scores_norm *0)

    # 5. retrieve the indices of the highest scores
    top_indices = np.argsort(final_scores)[::-1][:top_k]

    return [(final_scores[i], vector_scores_norm[i], lexical_scores_norm[i], tools_list[i]) for i in top_indices]

## 3. LLM Router Agent

In [16]:
class RouteDecision(BaseModel):
    # 1. On modifie la description pour forcer l'analyse avant le choix
    reasoning: str = Field(description="Step-by-step analysis comparing the user's request against the available tools before making a decision.")
    confidence: float = Field(description="Confidence level from 0.0 to 1.0")
    selected_tool: str = Field(description="The exact name of the tool. Return 'none' if no tool matches.")

# ----- AGENT LOGIC -----
def agent_router(user_prompt: str, relevant_tools: list, model_name: str) -> RouteDecision:
    """Passes the filtered tools and user prompt to the LLM to make the final routing decision."""
    
    # 2. Formatage des outils en texte clair (Markdown) au lieu d'un JSON brut
    tools_formatted = "\n".join([
        f"{t.get('name', 'Unknown')}: {t.get('description', '')} (Tags: {', '.join(t.get('tags', []))})" 
        for t in relevant_tools
    ])
    
    # 3. Prompt système enrichi avec des instructions strictes et des exemples (Few-Shot)
    system_prompt = f"""You are a Router Agent expert in medical and dental imaging (CBCT, IOS, MRI).
    Your role is to analyze the user's request and select the most relevant tool from the FILTERED list below.
    If none of these {len(relevant_tools)} tools fit perfectly, return 'none'.

    === FILTERED TOOLS ===
    {tools_formatted}

    === ROUTING GUIDELINES & EXAMPLES ===
    - Pay close attention to subtle differences. For example, if a user specifically asks for "batch processing" or "multiple scans", prioritize tools designed for batching (e.g., batchdentalseg).
    - If a user asks to "segment" or "split" specific teeth, ensure the tool handles instance segmentation (e.g., amasss_cli).
    - If the request is for registration, check if it's CBCT-to-CBCT, MRI-to-CBCT, or intraoral (IOS) and choose the specific tool accordingly.

    Carefully analyze the user's prompt step-by-step in the 'reasoning' field BEFORE selecting the tool. Output strictly matching the JSON schema.
    """
    
    try:
        response = chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=RouteDecision.model_json_schema(),
            options={"temperature": 0},
        )
        return RouteDecision.model_validate_json(response.message.content)
    except Exception as e:
        return RouteDecision(selected_tool="error", confidence=0.0, reasoning=f"Error: {str(e)}")

## 4. Data Loading and Pre-computation

In [17]:
# ----- LOAD FILES -----
with open(tools_list_path, 'r', encoding='utf-8') as f:
    tools_list = json.load(f)

with open(queries_list_path, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

# ----- ONE-TIME EMBEDDING CACHE and BM25 INDEX-----
t_start_setup = time.time()

# 1. Vector Cache
tool_matrix = precompute_tool_embeddings(tools_list)

# 2. Lexical Cache (BM25)
bm25 = precompute_bm25_cache(tools_list)
print(f"Embeddings and bm25 cached in {time.time() - t_start_setup:.2f}s")

Caching tool embeddings...
Embeddings and bm25 cached in 0.53s


## 5. Display Formatting Helpers

In [18]:
# ----- FORMATTING HELPER FUNCTIONS -----

def print_rag_block(index, total_queries, prompt, rag_icon, rag_results):
    """Handles the terminal output for the RAG search results."""
    print(f"\n{'='*65}")
    print(f"Prompt ({index}/{total_queries}) : '{prompt}'")
    print(f"{'-'*65}")
    print(f"I: RAG VECTOR SEARCH {rag_icon}")
    print(f"{'-'*65}")
    
    for i, (score, vec_score, bm25_score, tool) in enumerate(rag_results, start=1):
        tool_name = tool.get('name', 'unknown_name')
        print(f"{i} Score: {score:.4f} (Vec: {vec_score:.4f}, BM25: {bm25_score:.4f}) tools: {tool_name}")
    print(f"{'-'*65}")


def print_llm_block(decision, expected_tool, llm_status_icon, latency):
    """Handles the terminal output for the LLM routing decision."""
    print("II: LLM ROUTER DECISION")
    print(f"{'-'*65}")  
    print(f"Tool chosen: {decision.selected_tool} {llm_status_icon} ")
    print(f"Expected   : {expected_tool}")
    print(f"Confidence : {decision.confidence * 100:.2f}%")
    print(f"Latency    : {latency:.2f} seconds")
    print(f"Reasoning  : {decision.reasoning}")

## 6. Main Benchmark Loop

In [ ]:
# ----- BENCHMARK INITIALIZATION -----
results_detail = []
llm_correct_count = 0
rag_correct_count = 0
total_time = 0.0

limit = max(1, int(len(queries_list) * ratio_queries))
queries_to_run = queries_list[:limit]
total_queries = len(queries_to_run)

# ----- MAIN EXECUTION LOOP -----
for index, (prompt, expected_tool) in enumerate(queries_to_run, start=1):

    t0 = time.time()
    
    # 1. Execute RAG
    rag_results = retrieve_top_tools(prompt, tools_list, tool_matrix, bm25, top_k=4)
    relevant_tools = [tool for score, vec_score, bm25_score, tool in rag_results]    
    # Check RAG success & extract tool names
    rag_tool_names = [t.get('name', 'unknown') for t in relevant_tools]
    rag_hit = expected_tool in rag_tool_names
    
    if rag_hit: rag_correct_count += 1
    rag_hit_sign = "✅" if rag_hit else "❌"
    rag_prediction_str = f"{', '.join(rag_tool_names)} {rag_hit_sign}"

    print_rag_block(index, total_queries, prompt, rag_hit_sign, rag_results)

    # 2. Execute LLM Router
    decision = agent_router(prompt, relevant_tools, model)
    t1 = time.time()
    
    latency = t1 - t0
    total_time += latency
        
    # Check LLM success & trigger print
    llm_hit = decision.selected_tool == expected_tool
    if llm_hit: llm_correct_count += 1
    llm_hit_sign = "✅" if llm_hit else "❌"

    print_llm_block(decision, expected_tool, llm_hit_sign, latency)

    # 3. Save iteration data
    results_detail.append({
        "prompt": prompt,
        "expected_tool": expected_tool,
        "rag_prediction": rag_prediction_str,
        "llm_prediction": f"{decision.selected_tool} {llm_hit_sign}",
        "Confidence": decision.confidence,
        "Latency": round(latency, 4),
        "reasoning": decision.reasoning
    })


Prompt (1/140) : 'identify landmarks on CBCT scans'
-----------------------------------------------------------------
I: RAG VECTOR SEARCH ✅
-----------------------------------------------------------------
1 Score: 1.0000 (Vec: 1.0000, BM25: 0.8437) tools: ali_cbct
2 Score: 0.9094 (Vec: 0.9094, BM25: 1.0000) tools: semi_aso_cbct
3 Score: 0.8324 (Vec: 0.8324, BM25: 0.6272) tools: ali_ios
4 Score: 0.7610 (Vec: 0.7610, BM25: 0.2730) tools: areg_cbct
-----------------------------------------------------------------
II: LLM ROUTER DECISION
-----------------------------------------------------------------
Tool chosen: ali_cbct ✅ 
Expected   : ali_cbct
Confidence : 100.00%
Latency    : 2.27 seconds
Reasoning  : The user is asking to identify landmarks on CBCT scans, which suggests they are looking for anatomical reference points on cone-beam CT volumes for precise dental and maxillofacial measurements and analysis. This task can be achieved using the Automated Anatomic Landmarks Identificat

## 7. Save and Global Results

In [20]:
# ----- FINAL METRICS CALCULATION -----
total_queries = len(queries_to_run)

# Calculate both accuracies
rag_accuracy = (rag_correct_count / total_queries) * 100 if total_queries > 0 else 0.0
llm_accuracy = (llm_correct_count / total_queries) * 100 if total_queries > 0 else 0.0
avg_time = (total_time / total_queries) if total_queries > 0 else 0.0

# Structure the final output JSON report
summary = {
    "model": model,
    "metrics": {
        "total_queries": total_queries,
        "rag_correct":rag_correct_count,
        "rag_accuracy": round(rag_accuracy, 2),
        "llm_correct":llm_correct_count,
        "llm_accuracy": round(llm_accuracy, 2),
        "average_latency": round(avg_time, 4),
        "total_time": round(total_time, 4),
    },
    "details": results_detail,
}

# ----- EXPORT RESULTS -----
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# ----- FINAL SUMMARY OUTPUT -----
print("\n BENCHMARK COMPLETED ")
print(f"RAG Accuracy (Top): {rag_accuracy:.2f}% ({rag_correct_count}/{total_queries})")
print(f"LLM Accuracy (Exact): {llm_accuracy:.2f}% ({llm_correct_count}/{total_queries})")
print(f"Average Time:         {avg_time:.2f} seconds per query")
print(f"Report saved to:      {results_path.name}")


 BENCHMARK COMPLETED 
RAG Accuracy (Top): 90.00% (126/140)
LLM Accuracy (Exact): 76.43% (107/140)
Average Time:         1.97 seconds per query
Report saved to:      llama3-latest_tool_only_bm25.json
